# CARD External Validation (C4 Models)

Validates the C4-trained ML models on the CARD dataset per `CARD_VALIDATION_SPEC.md`.

- **No refitting**: C4 scaler and models are loaded and applied only (no fit on CARD).
- **45-feature alignment**: Same feature set and order as C4; AQ excluded from model input (used only for AQ >= 6 filter).
- **Pipeline**: Load CARD -> aggregate one row per participant (adult > adolescent > child) -> parse itemised scores with full-to-short mappings -> score SPQ/EQ/SQR/AQ -> demographics and derived -> AQ >= 6 exclusion -> drop adolescent/child-only -> balance 50/50 -> build feature matrix -> scale with C4 scaler (no refit) -> predict. Reports metrics at **threshold 0.5** and at **C4 F1-optimized thresholds** (from data_pipeline_recreation Experiment C) so the same decision rule as C4 is used.

In [ ]:
# Imports and config
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

_cwd = os.path.abspath(os.getcwd())
REPO_ROOT = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'notebooks' else _cwd
if not os.path.isdir(os.path.join(REPO_ROOT, 'models', 'cross_validation')):
    REPO_ROOT = _cwd
ARTIFACT_DIR = os.path.join(REPO_ROOT, 'models', 'cross_validation')
FEATURE_INFO_PATH = os.path.join(ARTIFACT_DIR, 'feature_info_original.json')
SCALER_PATH = os.path.join(ARTIFACT_DIR, 'scaler_original.joblib')
MODEL_NAMES = ['logistic_regression_original', 'random_forest_original', 'xgboost_original', 'lightgbm_original', 'gradient_boosting_original']
MODELS = {name.replace('_original', '').replace('_', ' ').title(): os.path.join(ARTIFACT_DIR, name + '.joblib') for name in MODEL_NAMES}

# CARD path: set CARD_PATH in env or below; otherwise first existing path is used
CARD_PATH = os.environ.get('CARD_PATH', '')
if not CARD_PATH or not os.path.exists(CARD_PATH):
    candidates = [
        os.path.join(REPO_ROOT, 'data', 'CARD_Nov2025(Sheet1).csv'),
        os.path.join(REPO_ROOT, 'data', 'CARD_Nov2025.xlsx'),
        os.path.expanduser('~/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/CARD_Nov2025(Sheet1).csv'),
        os.path.expanduser('~/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/CARD_Nov2025.xlsx'),
    ]
    for p in candidates:
        if os.path.exists(p):
            CARD_PATH = p
            break
if CARD_PATH:
    print(f'Using CARD: {CARD_PATH}')
else:
    print('CARD_PATH not set; define it in this cell (e.g. CARD_PATH = "path/to/CARD_Nov2025(Sheet1).csv")')
EXCEL_PASSWORD = os.environ.get('CARD_EXCEL_PASSWORD', '')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# Load CARD dataset (CSV or Excel)
if not CARD_PATH or not os.path.exists(CARD_PATH):
    raise FileNotFoundError(
        'CARD file not found. Set CARD_PATH in the previous cell to your CARD CSV/Excel path '
        '(e.g. data/CARD_Nov2025(Sheet1).csv or your OneDrive/data path).'
    )

def load_card(path):
    path = os.path.abspath(path)
    if path.lower().endswith(('.xlsx', '.xls')):
        try:
            return pd.read_excel(path, engine='openpyxl')
        except Exception as e:
            if 'encrypted' in str(e).lower() or 'BadZipFile' in str(type(e).__name__):
                try:
                    import msoffcrypto
                    import io
                    decrypted = io.BytesIO()
                    with open(path, 'rb') as f:
                        office_file = msoffcrypto.OfficeFile(f)
                        office_file.load_key(password=EXCEL_PASSWORD)
                        office_file.decrypt(decrypted)
                        decrypted.seek(0)
                    return pd.read_excel(decrypted, engine='openpyxl')
                except ImportError:
                    raise ImportError("pip install msoffcrypto-tool for encrypted Excel")
            raise
    for enc in ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except (UnicodeDecodeError, Exception):
            continue
    raise ValueError("Could not read CSV with common encodings")

df_card = load_card(CARD_PATH)
print(f"CARD loaded: {df_card.shape[0]} rows, {df_card.shape[1]} columns")

In [ ]:
# Detect key columns (case-insensitive)
def find_col(df, names, contains=None):
    for c in df.columns:
        c_lower = c.lower()
        if contains:
            if any(n in c_lower for n in names):
                return c
        else:
            if c_lower in [n.lower() for n in names]:
                return c
    return None

volunteer_id_col = find_col(df_card, ['volunteerid', 'volunteer id', 'userid']) or 'VolunteerID'
test_name_col = find_col(df_card, ['testname', 'test name']) or 'TestName'
itemised_col = find_col(df_card, ['itemised score', 'itemized score', 'itemisedscore']) or (None if 'Itemised Score' not in df_card.columns else 'Itemised Score')
if itemised_col is None:
    for c in df_card.columns:
        if 'itemis' in c.lower() and 'score' in c.lower():
            itemised_col = c
            break
diagnosis_col = find_col(df_card, ['asc diagnosis', 'diagnosis', 'asc']) or 'ASC diagnosis'
age_col = find_col(df_card, ['agewhentestcompleted', 'yearofbirth', 'age']) or None
sex_col = find_col(df_card, ['sex']) or 'Sex'
occupation_col = find_col(df_card, ['occupation']) or None

if volunteer_id_col not in df_card.columns:
    volunteer_id_col = [c for c in df_card.columns if 'volunteer' in c.lower() and 'id' in c.lower()]
    volunteer_id_col = volunteer_id_col[0] if volunteer_id_col else df_card.columns[0]
if itemised_col is None:
    raise ValueError("Could not find Itemised Score column. Columns: " + str(list(df_card.columns)))

print(f"VolunteerID: {volunteer_id_col}, TestName: {test_name_col}, Itemised: {itemised_col}, Diagnosis: {diagnosis_col}")

In [ ]:
# Item mappings (0-based) and test priority (adult=1, adolescent=2, child=3)
# NOTE: AQ-10 is NOT items 1-10 of AQ-50; it uses psychometrically-selected items.
# Source: Allison, Auyeung & Baron-Cohen (2012): AQ-50 items 5,20,27,28,31,32,36,37,41,45 (1-indexed)
# Converted to 0-indexed positions below.
ITEM_MAPPINGS = {
    'aq':  {'full_length': 50,  'short_items': [4, 19, 26, 27, 30, 31, 35, 36, 40, 44]},
    'eq':  {'full_length': 60,  'short_items': [13, 3, 8, 30, 27, 34, 11, 21, 17, 33]},
    'sqr': {'full_length': 75,  'short_items': [31, 15, 26, 8, 29, 32, 11, 24, 7, 6]},
    'spq': {'full_length': 92,  'short_items': [1, 20, 31, 34, 37, 57, 61, 72, 73, 87]},
}
TEST_PRIORITY = {
    'aq': 1, 'adolescent aq': 2, 'child aq': 3,
    'eq': 1, 'adolescent eq': 2, 'child eq': 3,
    'sq': 1, 'sqr': 1, 'adolescent sq': 2, 'child sq': 3,
    'spq': 1,
}

def test_priority(test_name):
    t = str(test_name).lower()
    for key, p in TEST_PRIORITY.items():
        if key in t:
            return p
    return 99

In [ ]:
def parse_itemised(itemised_str, test_name):
    if pd.isna(itemised_str):
        return {}
    items = [x.strip() for x in str(itemised_str).split(',') if x.strip()]
    t = str(test_name).lower()
    out = {}
    if 'aq' in t and 'adolescent' not in t and 'child' not in t:
        m = ITEM_MAPPINGS['aq']
        if len(items) >= m['full_length']:
            for i, idx in enumerate(m['short_items'], 1):
                if idx < len(items):
                    out[f'aq_{i}'] = items[idx]
        elif len(items) == 10:
            for i in range(1, 11):
                out[f'aq_{i}'] = items[i-1]
    elif 'eq' in t and 'adolescent' not in t and 'child' not in t:
        m = ITEM_MAPPINGS['eq']
        if len(items) >= m['full_length']:
            for i, idx in enumerate(m['short_items'], 1):
                if idx < len(items):
                    out[f'eq_{i}'] = items[idx]
        elif len(items) == 10:
            for i in range(1, 11):
                out[f'eq_{i}'] = items[i-1]
        elif len(items) >= 10:
            # CARD may use 40-item (or other) EQ; use first 10 as eq_1..eq_10
            for i in range(1, 11):
                out[f'eq_{i}'] = items[i-1]
    elif ('sq' in t or 'sqr' in t) and 'adolescent' not in t and 'child' not in t:
        m = ITEM_MAPPINGS['sqr']
        if len(items) >= m['full_length']:
            for i, idx in enumerate(m['short_items'], 1):
                if idx < len(items):
                    out[f'sqr_{i}'] = items[idx]
        elif len(items) == 10:
            for i in range(1, 11):
                out[f'sqr_{i}'] = items[i-1]
    elif 'spq' in t:
        m = ITEM_MAPPINGS['spq']
        if len(items) >= m['full_length']:
            for i, idx in enumerate(m['short_items'], 1):
                if idx < len(items):
                    out[f'spq_{i}'] = items[idx]
        elif len(items) == 10:
            for i in range(1, 11):
                out[f'spq_{i}'] = items[i-1]
    return out

df_card['_parsed'] = df_card.apply(lambda r: parse_itemised(r[itemised_col], r[test_name_col]), axis=1)
df_card['_priority'] = df_card[test_name_col].apply(test_priority)

In [ ]:
# Aggregate one row per participant (best priority per test type); keep only if at least one adult test
participants = []
eq_found_count = 0

# AQ branch counts (scope of AQ-50→AQ-10 mapping impact)
aq_full_count = 0      # len(items) >= 50 branch

aq_direct10_count = 0  # len(items) == 10 branch

aq_other_count = 0     # anything else / missing

for vid, grp in df_card.groupby(volunteer_id_col):
    if grp['_priority'].min() > 1:
        continue
    row0 = grp.iloc[0]
    demo = {c: row0[c] for c in df_card.columns if c not in ['_parsed', '_priority']}
    # Coalesce demographics from first non-null in group (row0 may be questionnaire-only)
    for col in [c for c in [volunteer_id_col, test_name_col, sex_col, 'AgeWhenTestCompleted', 'YearOfBirth', diagnosis_col, occupation_col] if c and c in grp.columns]:
        nonnull = grp[col].dropna()
        if len(nonnull) > 0 and pd.isna(demo.get(col)):
            demo[col] = nonnull.iloc[0]
    merged = {}
    for test_type in ['aq', 'eq', 'sqr', 'spq']:
        sub = grp[grp[test_name_col].astype(str).str.lower().str.contains(test_type if test_type != 'sqr' else 'sq', na=False)]
        if len(sub) == 0:
            continue
        best = sub.loc[sub['_priority'].idxmin()]

        # Report how many participants use the AQ-50 mapping vs direct AQ-10.
        if test_type == 'aq':
            try:
                raw_itemised = best[itemised_col]
                items = [x.strip() for x in str(raw_itemised).split(',') if x.strip()] if pd.notna(raw_itemised) else []
                n_items = len(items)
                if n_items >= ITEM_MAPPINGS['aq']['full_length']:
                    aq_full_count += 1
                elif n_items == 10:
                    aq_direct10_count += 1
                else:
                    aq_other_count += 1
            except Exception:
                aq_other_count += 1

        if test_type == 'eq' and len(best['_parsed']) > 0:
            eq_found_count += 1
        for k, v in best['_parsed'].items():
            merged[k] = v
    for k, v in merged.items():
        demo[k] = v
    participants.append(demo)

df_agg = pd.DataFrame(participants)
print(f"Aggregated: {len(df_agg)} participants (adult questionnaires only)")
print(f"  Participants with EQ data: {eq_found_count}")
print(
    "  AQ item source (per participant, chosen adult AQ row): "
    f"AQ-50+ mapped={aq_full_count}, direct AQ-10 len==10={aq_direct10_count}, other/unknown={aq_other_count}"
)

eq_cols_check = [c for c in df_agg.columns if c.startswith('eq_')]
if eq_cols_check:
    eq_nonnull = df_agg[eq_cols_check].notna().any(axis=1).sum()
    print(f"  Rows with at least one eq_* column: {eq_nonnull}")

In [ ]:
# Score SPQ-10, EQ-10, SQR-10, AQ-10 (C4 rules)
# CARD stores all questionnaires on a 0-3 scale, but AQ appears reversed (0/1 = agree, 2/3 = disagree).
# We convert AQ items to the C4-style 1-4 semantics via (3 - x) so that 0/1 -> 3/2 (agree) and 2/3 -> 1/0 (disagree).
for i in range(1, 11):
    for prefix in ['spq', 'eq', 'sqr', 'aq']:
        c = f'{prefix}_{i}'
        if c not in df_agg.columns:
            df_agg[c] = np.nan
        else:
            df_agg[c] = pd.to_numeric(df_agg[c], errors='coerce')

# Diagnostic: check raw values before scoring
eq_cols = [f'eq_{i}' for i in range(1, 11)]
eq_raw = df_agg[eq_cols].stack().dropna()
if len(eq_raw) == 0:
    print('  EQ: No non-NaN values found - EQ items may not be parsed/aggregated')
else:
    print(f'  EQ raw values: min={eq_raw.min():.0f}, max={eq_raw.max():.0f}, mean={eq_raw.mean():.2f}, count={len(eq_raw)}')

aq_cols = [f'aq_{i}' for i in range(1, 11)]
aq_raw = df_agg[aq_cols].stack().dropna()
if len(aq_raw) == 0:
    print('  AQ: No non-NaN values found - AQ items may not be parsed/aggregated')
else:
    vc = aq_raw.value_counts().sort_index()
    print(f"  AQ raw values (pre-flip): min={aq_raw.min():.0f}, max={aq_raw.max():.0f}, mean={aq_raw.mean():.2f}, count={len(aq_raw)}")
    print(f"  AQ raw value counts (expect only 0,1,2,3): {vc.to_dict()}")

# Flip AQ 0-3 coding so that 0/1 = disagree, 2/3 = agree, matching C4 _bin03 semantics.
for i in range(1, 11):
    c = f'aq_{i}'
    if c in df_agg.columns:
        df_agg[c] = df_agg[c].apply(lambda x: (3 - float(x)) if pd.notna(x) else np.nan)
print('  ✅ Applied AQ flip: df_agg[aq_i] <- 3 - df_agg[aq_i] for CARD 0-3 scale.')

# SPQ: CARD 0-3 = already 0-3 per item (same as C4). If 1-4 present, convert 4-raw.
spq_cols = [f'spq_{i}' for i in range(1, 11)]
spq_vals = df_agg[spq_cols].stack().dropna()
if len(spq_vals) > 0 and spq_vals.max() > 3:
    for i in range(1, 11):
        df_agg[f'spq_{i}'] = 4 - df_agg[f'spq_{i}']
df_agg['spq_total'] = df_agg[spq_cols].sum(axis=1)

# Binary agree/disagree for 0-3: agree=2,3 -> 1; disagree=0,1 -> 0 (reverse items swap)
def _bin03(x, reverse):
    if pd.isna(x): return np.nan
    x = float(x)
    agree = 2 <= x <= 3
    disagree = 0 <= x <= 1
    return (1 if disagree else (0 if agree else np.nan)) if reverse else (1 if agree else (0 if disagree else np.nan))

# EQ: reverse item 3 (fix lambda closure by capturing i explicitly)
for i in range(1, 11):
    rev = (i == 3)
    df_agg[f'eq_{i}'] = df_agg[f'eq_{i}'].apply(lambda x, r=rev: _bin03(x, r))
df_agg['eq_total'] = df_agg[eq_cols].sum(axis=1)

# SQR: reverse 2,4,6,8,10
sqr_rev = {2,4,6,8,10}
for i in range(1, 11):
    df_agg[f'sqr_{i}'] = df_agg[f'sqr_{i}'].apply(lambda x, r=(i in sqr_rev): _bin03(x, r))
df_agg['sqr_total'] = df_agg[[f'sqr_{i}' for i in range(1, 11)]].sum(axis=1)

# AQ: reverse 2,3,4,5,6,9 (for filter only)
aq_rev = {2,3,4,5,6,9}
for i in range(1, 11):
    df_agg[f'aq_{i}'] = df_agg[f'aq_{i}'].apply(lambda x, r=(i in aq_rev): _bin03(x, r))
df_agg['aq_total'] = df_agg[[f'aq_{i}' for i in range(1, 11)]].sum(axis=1)

print("Totals (CARD 0-3 scale):", df_agg[['spq_total','eq_total','sqr_total','aq_total']].describe().loc[['mean','min','max']])
print("  (SPQ: 10 items x 0-3 = 0-30, same as C4. EQ/SQR/AQ: 10 items binary 0/1 = 0-10. SQR max 9 in data = no participant had all 10 items scored 1.)")

In [ ]:
# Demographics and derived variables
# Age: prefer AgeWhenTestCompleted; if only YearOfBirth, convert to age (ref_year - YOB)
ref_year = 2024
age_ser = None
age_source = None
if 'AgeWhenTestCompleted' in df_agg.columns:
    age_ser = pd.to_numeric(df_agg['AgeWhenTestCompleted'], errors='coerce')
    age_source = 'AgeWhenTestCompleted'
if age_ser is None and 'YearOfBirth' in df_agg.columns:
    yob = pd.to_numeric(df_agg['YearOfBirth'], errors='coerce')
    age_ser = (ref_year - yob).clip(lower=0, upper=120)
    age_source = 'YearOfBirth (age = ref_year - YOB)'
if age_ser is None and age_col and age_col in df_agg.columns:
    raw = pd.to_numeric(df_agg[age_col], errors='coerce')
    if raw.max() <= 120 and raw.min() >= 0:
        age_ser = raw
        age_source = age_col
    else:
        age_ser = (ref_year - raw).clip(lower=0, upper=120)
        age_source = f'{age_col} (treated as YOB)'
df_agg['age'] = age_ser.fillna(age_ser.median()).clip(lower=0, upper=120) if age_ser is not None else 30
if age_source:
    print(f"Age source: {age_source}; range {df_agg['age'].min():.0f} - {df_agg['age'].max():.0f}, mean {df_agg['age'].mean():.1f}")

# Sex: CARD column 'Sex' has values M and F. C4 uses 1=male, 2=female, 3=other, 4=prefer not to say; sex_num = 0,1,2,3.
if sex_col and sex_col in df_agg.columns:
    raw = df_agg[sex_col].astype(str).str.strip().str.lower()
    str_to_num = {'male': 1, 'female': 2, 'm': 1, 'f': 2, 'other': 3, 'prefer not to say': 4, 'prefer not to say ': 4}
    df_agg['sex'] = raw.map(str_to_num)
    numeric_sex = pd.to_numeric(df_agg[sex_col], errors='coerce')
    df_agg['sex'] = df_agg['sex'].fillna(numeric_sex).fillna(0).astype(int).clip(0, 4)
else:
    df_agg['sex'] = 0
df_agg['sex_num'] = df_agg['sex'].map({1: 0, 2: 1, 3: 2, 4: 3}).fillna(0).astype(int)
print(f"Sex (1=M,2=F,3=O,4=PNS): {df_agg['sex'].value_counts().sort_index().to_dict()}; sex_num: {df_agg['sex_num'].value_counts().sort_index().to_dict()}")

df_agg['age_group_19-30'] = ((df_agg['age'] >= 19) & (df_agg['age'] <= 30)).astype(int)
df_agg['age_group_31-45'] = ((df_agg['age'] >= 31) & (df_agg['age'] <= 45)).astype(int)
df_agg['age_group_46-60'] = ((df_agg['age'] >= 46) & (df_agg['age'] <= 60)).astype(int)
df_agg['age_group_61+'] = (df_agg['age'] >= 61).astype(int)
df_agg['sqrt_age'] = np.sqrt(df_agg['age'].clip(lower=0))
df_agg['d_score'] = df_agg['sqr_total'].fillna(0) - df_agg['eq_total'].fillna(0)
df_agg['age_x_eq'] = df_agg['age'] * df_agg['eq_total'].fillna(0)
denom = df_agg['sqr_total'].replace(0, np.nan) + 1e-8
df_agg['eq_sqr_ratio'] = (df_agg['eq_total'] / denom).replace([np.inf, -np.inf], np.nan).fillna(0)

stem_keywords = 'science technology engineering math computer software data research'.split()
if occupation_col and occupation_col in df_agg.columns:
    s = df_agg[occupation_col].astype(str).str.lower()
    df_agg['is_stem_occupation'] = s.str.contains('|'.join(stem_keywords), case=False, na=False).astype(int)
else:
    df_agg['is_stem_occupation'] = 0

# Target from ASC diagnosis
if diagnosis_col not in df_agg.columns:
    diagnosis_col = [c for c in df_agg.columns if 'diagnos' in c.lower() or 'asc' in c.lower()]
    diagnosis_col = diagnosis_col[0] if diagnosis_col else None
if diagnosis_col:
    d = df_agg[diagnosis_col]
    if pd.api.types.is_numeric_dtype(d):
        df_agg['autism_target'] = (d == 1).astype(int)
    else:
        df_agg['autism_target'] = d.astype(str).str.lower().str.contains('autism|asc|asd|yes|1|true', na=False).astype(int)
else:
    df_agg['autism_target'] = 0
print("autism_target:", df_agg['autism_target'].value_counts().to_dict())

In [ ]:
# AQ >= 6 exclusion (autism cases only), then 50/50 balance
n_before = len(df_agg)
cases_before = (df_agg['autism_target'] == 1).sum()
controls_before = (df_agg['autism_target'] == 0).sum()
print(f"Before AQ filter: {n_before} rows (autism={cases_before}, non-autism={controls_before})")

# Sanity checks for AQ-10 (before filtering)
try:
    mean_aut = df_agg.loc[df_agg['autism_target'] == 1, 'aq_total'].mean()
    mean_non = df_agg.loc[df_agg['autism_target'] == 0, 'aq_total'].mean()
    pct_ge6 = 100.0 * (df_agg.loc[df_agg['autism_target'] == 1, 'aq_total'] >= 6).mean()
    print(f"AQ-10 sanity (before filter): mean autism={mean_aut:.2f}/10, mean non-autism={mean_non:.2f}/10, % autism >=6 = {pct_ge6:.1f}%")
except Exception as e:
    print('AQ-10 sanity check failed:', e)

df_agg = df_agg.copy()
aq_ok = (df_agg['autism_target'] != 1) | (df_agg['aq_total'].fillna(-1) >= 6)
dropped = (~aq_ok).sum()
df_agg = df_agg[aq_ok].reset_index(drop=True)

cases = df_agg[df_agg['autism_target'] == 1]
controls = df_agg[df_agg['autism_target'] == 0]
n_cases = len(cases)
n_controls = len(controls)
print(f"After AQ >= 6 exclusion: {len(df_agg)} rows (autism={n_cases}, non-autism={n_controls}); removed {dropped} autism cases with AQ < 6")

target_n = min(n_cases, n_controls)
if target_n == 0:
    raise ValueError("No cases or no controls after AQ filter")
controls_samp = controls.sample(n=target_n, random_state=RANDOM_SEED)
cases_samp = cases.sample(n=target_n, random_state=RANDOM_SEED) if n_cases > target_n else cases
df_agg = pd.concat([controls_samp, cases_samp], ignore_index=True)
print(f"Balanced 50/50: {len(df_agg)} rows ({target_n} autism, {target_n} non-autism)")

# Show one row per participant: head and tail of key columns
key_cols = [c for c in ['age', 'sex_num', 'autism_target', 'spq_total', 'eq_total', 'sqr_total', 'aq_total'] if c in df_agg.columns]
if volunteer_id_col in df_agg.columns:
    key_cols = [volunteer_id_col] + key_cols

print("\nOne row per participant (head of balanced df_agg, first 20 rows):")
print(df_agg[key_cols].head(20).to_string())

print("\nOne row per participant (tail of balanced df_agg, last 20 rows):")
print(df_agg[key_cols].tail(20).to_string())

print(f"\n(Each row above is one participant; total rows = {len(df_agg)})")

In [ ]:
# Build 45-feature matrix aligned to C4 schema
with open(FEATURE_INFO_PATH, 'r') as f:
    feature_info = json.load(f)
c4_feature_names = feature_info['feature_names']

print("All 45 C4 features used (in order):")
for i, name in enumerate(c4_feature_names, 1):
    print(f"  {i:2}. {name}")

X_card = pd.DataFrame(index=df_agg.index)
for name in c4_feature_names:
    X_card[name] = df_agg[name] if name in df_agg.columns else 0
X_card = X_card[c4_feature_names]
X_card = X_card.fillna(0)
for col in X_card.columns:
    X_card[col] = pd.to_numeric(X_card[col], errors='coerce').fillna(0)
X_card = X_card.replace([np.inf, -np.inf], 0)
print(f"\nX_card shape: {X_card.shape}")

print("\nKey feature ranges (sanity check):")
for c in ['age', 'spq_total', 'eq_total', 'sqr_total', 'age_x_eq', 'sqrt_age']:
    if c in X_card.columns:
        print(f"  {c}: min={X_card[c].min():.3f}, max={X_card[c].max():.3f}, mean={X_card[c].mean():.3f}")

# Feature breakdown by group
groups = {
    'demographics': ['age', 'sex', 'sex_num'],
    'spq_items': [f'spq_{i}' for i in range(1, 11)],
    'eq_items': [f'eq_{i}' for i in range(1, 11)],
    'sqr_items': [f'sqr_{i}' for i in range(1, 11)],
    'totals_derived': ['spq_total', 'eq_total', 'sqr_total', 'd_score', 'sqrt_age', 'age_x_eq', 'eq_sqr_ratio'],
    'occupation': ['is_stem_occupation'],
    'age_groups': ['age_group_19-30', 'age_group_31-45', 'age_group_46-60', 'age_group_61+'],
}
for grp_name, feats in groups.items():
    feats = [f for f in feats if f in X_card.columns]
    if not feats:
        continue
    sub = X_card[feats]
    n_nonzero = (sub != 0).any(axis=1).sum()
    ranges = sub.agg(['min', 'max', 'mean']).round(3)
    print(f"\n{grp_name} ({len(feats)}): {feats[0]}..{feats[-1] if len(feats)>1 else feats[0]}")
    print(f"  rows with any non-zero: {n_nonzero}/{len(X_card)}")
    print(f"  min/max/mean: {sub.min().min():.3f} / {sub.max().max():.3f} / {sub.values.mean():.3f}")

print("\nSummary: all 45 C4 features (min, max, mean):")
for c in X_card.columns:
    s = X_card[c]
    print(f"  {c}: min={s.min():.3f}, max={s.max():.3f}, mean={s.mean():.3f}")

# Save aligned CARD dataset for Study 1/2/3 (same path used in study notebooks)
card_aligned_path = os.path.join(REPO_ROOT, 'data', 'processed', 'card_aligned.csv')
os.makedirs(os.path.dirname(card_aligned_path), exist_ok=True)
card_out = X_card.copy()
card_out['autism_target'] = df_agg['autism_target'].values
card_out['age'] = df_agg['age'].values
card_out['aq_total'] = df_agg['aq_total'].values
# Parse comorbidities from any column in df_agg (CARD may use different column names)
combined = df_agg.astype(str).agg(' '.join, axis=1).str.lower()
card_out['has_adhd'] = (combined.str.contains('adhd', na=False) | combined.str.contains('attention', na=False) | combined.str.contains('hyperactiv', na=False)).astype(int).values
card_out['has_anxiety'] = (combined.str.contains('anxiety', na=False) | combined.str.contains('anxious', na=False)).astype(int).values
card_out['has_depression'] = (combined.str.contains('depression', na=False) | combined.str.contains('depress', na=False)).astype(int).values
card_out.to_csv(card_aligned_path, index=False)
print(f"\nSaved CARD aligned dataset to {card_aligned_path} ({len(card_out)} rows, 45 features + autism_target, age, aq_total, has_adhd/has_anxiety/has_depression)")
print(f"  Comorbidity counts: has_adhd={card_out['has_adhd'].sum()}, has_anxiety={card_out['has_anxiety'].sum()}, has_depression={card_out['has_depression'].sum()}")

In [ ]:
# Load scaler and C4-optimized thresholds (F1-optimized on C4 test set)
scaler = joblib.load(SCALER_PATH)
X_card_scaled = scaler.transform(X_card)
y_true = df_agg['autism_target'].values
print(f"y_true: {y_true.sum()} autism, {len(y_true) - y_true.sum()} non-autism\n")

THRESHOLDS_PATH = os.path.join(ARTIFACT_DIR, 'optimal_thresholds.json')
threshold_csv_path = os.path.join(REPO_ROOT, 'data', 'processed', 'threshold_optimization_results.csv')
c4_thresholds = {}
if os.path.exists(THRESHOLDS_PATH):
    try:
        with open(THRESHOLDS_PATH, 'r') as f:
            raw = json.load(f)
        for k, v in raw.items():
            c4_thresholds[k.replace(' ', '').lower()] = float(v)
        print(f"Loaded C4 F1-optimized thresholds from {THRESHOLDS_PATH}")
    except (json.JSONDecodeError, ValueError, TypeError):
        print(f"optimal_thresholds.json empty or invalid; using 0.5 for all models.")
if not c4_thresholds and os.path.exists(threshold_csv_path):
    tdf = pd.read_csv(threshold_csv_path, index_col=0)
    if 'f1_threshold' in tdf.columns:
        for k in tdf.index:
            c4_thresholds[str(k).replace(' ', '').lower()] = float(tdf.loc[k, 'f1_threshold'])
        print(f"Loaded C4 F1-optimized thresholds from {threshold_csv_path}")
if not c4_thresholds:
    print("No C4 optimal_thresholds.json or threshold_optimization_results.csv found; using 0.5 for all models.")

def get_threshold(display_name):
    key = display_name.replace(' ', '').lower()
    return c4_thresholds.get(key, 0.5)

results_05 = []
results_c4 = []
for display_name, path in MODELS.items():
    if not os.path.exists(path):
        continue
    model = joblib.load(path)
    if type(model).__name__ == 'LGBMClassifier':
        X_in = pd.DataFrame(X_card_scaled, columns=X_card.columns)
    else:
        X_in = X_card_scaled
    y_proba = model.predict_proba(X_in)[:, 1]
    thresh_c4 = get_threshold(display_name)
    y_pred_05 = (y_proba >= 0.5).astype(int)
    y_pred_c4 = (y_proba >= thresh_c4).astype(int)
    for y_pred, res_list in [(y_pred_05, results_05), (y_pred_c4, results_c4)]:
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        res_list.append({'Model': display_name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1})
    try:
        auc = roc_auc_score(y_true, y_proba)
    except Exception:
        auc = float('nan')
    results_05[-1]['ROC-AUC'] = auc
    results_c4[-1]['ROC-AUC'] = auc
    print(f"{display_name}: threshold_0.5={0.5}, threshold_C4={thresh_c4:.4f}; pred_0.5: 0={ (y_pred_05==0).sum() } 1={ (y_pred_05==1).sum() }; pred_C4: 0={ (y_pred_c4==0).sum() } 1={ (y_pred_c4==1).sum() }; proba mean={y_proba.mean():.3f}")

print("\n--- Metrics at default threshold 0.5 ---")
print(pd.DataFrame(results_05).to_string(index=False))
print("\n--- Metrics at C4 F1-optimized thresholds (same decision rule as C4) ---")
print(pd.DataFrame(results_c4).to_string(index=False))

## Summary

- Loaded CARD dataset and aggregated to one row per participant (adult questionnaires preferred)
- Parsed itemised scores with full-to-short mappings (AQ-50→AQ-10, EQ-60→EQ-10, SQ-R-75→SQ-R-10, SPQ-92→SPQ-10)
- Scored all questionnaires per C4 rules (SPQ: 4-raw; EQ/SQR/AQ: binary with reverse items)
- Applied AQ >= 6 exclusion to autism cases, dropped adolescent/child-only participants, balanced 50/50
- Built 45-feature matrix aligned to C4 schema (AQ excluded from model input)
- Scaled with C4 scaler (no refitting); predictions at threshold 0.5 and at C4 F1-optimized thresholds

Results above: first table = default 0.5 threshold; second table = C4 F1-optimized thresholds (from data_pipeline_recreation Experiment C). Using C4 thresholds is not leakage (they were chosen on C4, not CARD) and matches how C4 was evaluated.